<a href="https://colab.research.google.com/github/girish17work/Data-Analysis/blob/main/Unknown_Dataset_4/Unknown_Dataset_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
DATASET_URL = 'https://drive.google.com/uc?export=download&id=1cHGfVASFltupF3uLF3LoYXiwftoLpk3u'
df = pd.read_csv(DATASET_URL)
df.head()

,CRIM,ZN,INDUS,POLINDEX,RM,AGE,DIS,HIGHWAYCOUNT,TAX,PTRATIO,IMM,BPL,PRICE,RIVERSIDE
0,1.026769,1.429034,7.851300,1.134216,6.0,42.0,5.251911,5,279.201277,20.689586,398.811960,10.461456,22.991633,NO
1,0.848089,0.255543,6.263434,1.245993,7.0,63.0,4.305546,8,307.444529,17.465398,377.153649,11.619690,24.551055,NO
2,10.925905,0.441022,18.322960,2.824833,8.0,-2.0,2.409495,25,666.492973,20.351601,387.061355,19.366070,15.875346,NO
3,0.559027,1.041175,11.114920,0.794952,6.0,9.0,6.898669,4,305.514181,19.787314,391.778647,6.206820,23.007756,NO
4,0.905063,81.167963,3.673369,1.029030,8.0,20.0,10.246463,1,315.913960,17.360439,395.833166,10.827105,21.503177,NO


In [3]:
print(f'The number of columns in the dataset is {df.shape[1]}')

The number of columns in the dataset is 14


In [4]:
print(f'The median of house price is {df['PRICE'].median()}')

The median of house price is 22.634566447766908


In [5]:
print(f'The number of houses with 9 or more rooms is {(df['RM']>=9).sum()}')

The number of houses with 9 or more rooms is 318


In [12]:
print(f'The average price of 10 least expensive house is {df['PRICE'].nsmallest(10).mean()}')

The average price of 10 least expensive house is 7.278891822010996


In [13]:
df['RM'].unique()

array([ 6.,  7.,  8.,  9., 10., -1.,  5.,  4., 11.])

In [14]:
df['AGE'].unique()

array([ 42.,  63.,  -2.,   9.,  20.,  95.,  35.,  97.,  36.,  78.,  99.,
        96.,  19.,  73.,  86.,  37.,  76.,  50.,  32.,  92., 101.,   8.,
        94.,  84.,  47.,  43.,  65.,  46.,  59.,  74.,  93.,  80.,  98.,
       102.,  56., 100.,  89.,  53.,  18.,  71.,  62.,  83.,  66.,   7.,
        77.,  91.,  90.,  58.,  57.,  60.,  87.,  75.,  34.,  85.,  48.,
        33.,  38.,  30.,  23.,  55.,  11.,  14.,  45.,  41.,  88.,  16.,
        12.,  79.,  67.,  39.,  49.,  61.,  70.,  54.,  10.,  22.,  72.,
        15.,  51.,  81.,  31.,  52.,  44.,  28.,  17.,  40.,  29.,  69.,
        64.,  24.,  82.,  26.,  68.,  27.,  13., 103.,  21.,  25.,   6.,
         5.,   3.,   4.])

In [15]:
df['RIVERSIDE'].unique()

array(['NO', 'UNKNOWN', 'YES'], dtype=object)

In [16]:
missing_values_in_rm = (df['RM']==-1).sum()
missing_values_in_age = (df['AGE']==-2).sum()
missing_values_in_riverside = (df['RIVERSIDE']=='UNKNOWN').sum()

print(f'The combined missing values count is {missing_values_in_rm + missing_values_in_age + missing_values_in_riverside}')

The combined missing values count is 178


In [21]:
import numpy as np

print(f'Average age is {df['AGE'].replace(-2, np.nan).mean()}')

Average age is 66.86582278481012


In [26]:
print(f'The percentage of house with unknown riverside is {((df['RIVERSIDE']=='UNKNOWN').mean())*100}')

The percentage of house with unknown riverside is 2.1999999999999997


In [27]:
print(f'The number of houses not on the riverside and older than 60 years is {((df['RIVERSIDE']=='NO') & (df['AGE']>60)).sum()}')

The number of houses not on the riverside and older than 60 years is 2088


In [28]:
print(f'The number of houses that are near to exactly 3, 4 or 5 highways is {(df['HIGHWAYCOUNT'].isin([3,4,5])).sum()}')

The number of houses that are near to exactly 3, 4 or 5 highways is 1742


In [55]:
bins = [-np.inf, 15, 25, 35, np.inf]
labels = ['Category 1', 'Category 2', 'Category 3', 'Category 4']

df['CATEGORY'] = pd.cut(df['PRICE'], bins=bins, labels=labels, right=False)
print(df['CATEGORY'].value_counts().idxmin())

Category 1


In [56]:
from sklearn.model_selection import train_test_split

df['RM'] = df['RM'].replace(-1, np.nan)
df['AGE'] = df['AGE'].replace(-2, np.nan)
df['RIVERSIDE'] = df['RIVERSIDE'].replace('UNKNOWN', np.nan)

X = df.drop(columns=['PRICE'])
y = df['PRICE']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

In [57]:
print(f'The number of samples in the test set is {X_test.shape[0]}')

The number of samples in the test set is 1000


In [58]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = df.drop(columns=['CATEGORY'])

standard_scale_features = ['CRIM', 'POLINDEX']
min_max_scale_features = ['ZN', 'INDUS', 'DIS', 'HIGHWAYCOUNT', 'TAX', 'PTRATIO', 'IMM', 'BPL']

rm = Pipeline([('impute', SimpleImputer(strategy='mean')), ('scale', MinMaxScaler())])
age = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
riverside = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[
    ('standard', StandardScaler(), standard_scale_features),
    ('minmax', MinMaxScaler(), min_max_scale_features),
    ('rm', rm, ['RM']),
    ('age', age, ['AGE']),
    ('riverside', riverside, ['RIVERSIDE'])
]).set_output(transform='pandas')

In [59]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

print(f'The number of features after performing the transformation is {X_train.shape[1]}')

The number of features after performing the transformation is 13


In [63]:
print(f'The mean of the transformed training data is {X_train.to_numpy().mean()}')

The mean of the transformed training data is 0.28842989630040944


In [68]:
DATASET_URL = 'https://drive.google.com/uc?export=download&id=1OtwemNuw0qilXAutyuZEjOscZVKuD7Tv'
df = pd.read_csv(DATASET_URL)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0.087472,0.002829,1.768235,0.188396,0.117071,0.148148,0.478165,0.720443,0.967195,0.281563,0.428571,0.959596,1.0,0.0,23.358097
1,0.378379,0.022079,1.115629,0.091974,0.066089,0.851852,0.911759,0.785321,0.885001,0.424648,0.285714,0.868687,0.0,1.0,17.268768
2,0.066901,0.003828,-0.536262,0.221188,0.255671,0.296296,0.228024,0.406472,0.980184,0.274376,0.428571,0.767677,0.0,1.0,27.776974
3,0.140645,0.011132,1.323366,0.422514,0.153103,0.148148,0.410679,0.200319,0.861371,0.305006,0.142857,0.848485,0.0,1.0,16.121960
4,0.144225,0.204918,-0.930790,0.148694,0.177490,0.259259,0.146832,0.111429,0.983448,0.286322,0.285714,0.616162,1.0,0.0,23.129426


In [70]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1:]

In [71]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [73]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

ridge = Ridge(alpha=5, solver='sag', tol=1e-4, random_state=42)
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_test)
print(f'R2 score: {r2_score(y_test, y_pred)}')

R2 score: 0.6639511913668861


In [82]:
print(f'Index of most important feature: {abs(ridge.coef_).argmax()}')

Index of most important feature: 9


In [84]:
print(f'Index of least important feature: {np.argmin(np.abs(ridge.coef_))}')

Index of least important feature: 2


In [86]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV

estimator = SGDRegressor(random_state=42)

param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'alpha': [1e-4, 1e-3, 1e-2, 1e-1, 1],
    'tol': [1e-3, 1e-2]
}

grid_search_cv = GridSearchCV(estimator=estimator, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search_cv.fit(X_train, y_train)

best_estimator = grid_search_cv.best_estimator_

In [91]:
print(f'Best alpha is {grid_search_cv.best_params_['alpha']}')

Best alpha is 0.0001


In [92]:
from sklearn.metrics import root_mean_squared_error

y_pred = best_estimator.predict(X_test)

print(f'Root mean squared error is {root_mean_squared_error(y_test, y_pred)}')

Root mean squared error is 5.126302501903342


In [99]:
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([('pca', PCA()), ('lasso', Lasso())])

param_grid = {
    'pca__n_components': [0.85, 0.95],
    'lasso__alpha': [5, 1, 0.1, 0.01]
}

grid_search_cv = GridSearchCV(pipeline, param_grid=param_grid, cv=5, n_jobs=-1, scoring='neg_mean_absolute_error')
grid_search_cv.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('pca', PCA()), ('lasso', Lasso())]),
             n_jobs=-1,
             param_grid={'lasso__alpha': [5, 1, 0.1, 0.01],
                         'pca__n_components': [0.85, 0.95]},
             scoring='neg_mean_absolute_error')

In [100]:
from sklearn.metrics import mean_absolute_error

y_pred = grid_search_cv.predict(X_test)

print(f'Mean absolute error: {mean_absolute_error(y_test, y_pred)}')

Mean absolute error: 3.865036507995


In [120]:
print(f'Percentage of the variance explained by the first principal component is {(grid_search_cv.best_estimator_.named_steps['pca'].explained_variance_ratio_[0])*100}')

Percentage of the variance explained by the first principal component is 69.9021494220279


In [123]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

pipeline = Pipeline([('polynomial', PolynomialFeatures(interaction_only=False, degree=2)), ('lasso', Lasso(alpha=0.5, warm_start=True, random_state=0))])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(f'Mean Absolute Error: {mean_absolute_error(y_test, y_pred)}')

Mean Absolute Error: 4.55907406372981


In [129]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

estimator = LinearRegression()

rfe = RFE(estimator=estimator, n_features_to_select=X_train.shape[1]-2)
rfe.fit(X_train, y_train)

print(np.where(~rfe.support_)[0])


[2 3]
